# Lab 06 · Sources

A **source** is anything with a `block(frames)` method that gives back the next `frames` samples. A mixer is a source made of sources. In this lab everything is a source: waveforms, noises, effects, even the parameters of other sources.

Each source keeps its own **state** between one block and the next: a phase, the last few samples, a buffer. Most exercises are about finding what that state is.

A few things to know:

- `live(source)` plays a source, and stops whatever was playing before. `stop()` stops it.
- To wait between two changes, use `await asyncio.sleep(seconds)`. It works in a local Jupyter and in the browser, where `time.sleep` would stop the sound too.
- Keep the volume low: every source here stays below 0.3. The stream clips at ±1 in any case.
- In the browser, run the setup and press **▶ audio** once.

## Setup

The tools for notes and scales, and the sources built in class: `Sine`, `Square`, `Voice` (with `done`), `Mixer`, `stream_of`, and `live` and `stop`.

In [2]:
import sounddevice as sd
import numpy as np
import asyncio

SR = 48000
rng = np.random.default_rng()

In [3]:
from IPython.display import Audio
from dataclasses import dataclass

def play(wave):
    return Audio(wave, rate=SR, normalize=False, autoplay=False)

def pluck(f, dur_s=1.0, tau=0.3, amplitude=0.25):
    t = np.arange(round(dur_s * SR)) / SR
    env = np.exp(-t / tau)
    wave = np.sin(2 * np.pi * f * t)
    return amplitude * env * wave

def mix_sounds(sounds):                      # a sound is (onset_s, wave)
    dur_s = max([onset_s + len(wave) / SR for onset_s, wave in sounds])
    out = np.zeros(round(dur_s * SR))
    for onset_s, wave in sounds:
        onset = round(onset_s * SR)
        out[onset:onset + len(wave)] += wave
    return out

note_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

class Pitch:
    def __init__(self, value):
        if isinstance(value, str):           # a name like "C4": the octave last, the note before
            value = 12 * (int(value[-1]) + 1) + note_names.index(value[:-1])
        self.midi = value

    @property
    def f(self):                             # the frequency in Hz
        return 440 * 2 ** ((self.midi - 69) / 12)

    def __add__(self, semitones):
        if isinstance(semitones, Pitch):
            raise TypeError("Add semitones, not a pitch.")
        return Pitch(self.midi + semitones)

    def __sub__(self, other):
        if isinstance(other, Pitch):
            return self.midi - other.midi    # an interval
        return Pitch(self.midi - other)      # a lower pitch

    def __eq__(self, other):
        return isinstance(other, Pitch) and self.midi == other.midi

    def __repr__(self):
        return note_names[self.midi % 12] + str(self.midi // 12 - 1)

@dataclass # automatically generates: __init__(self, events, length), __repr__, __eq__
class Clip:
    events: list      # (onset, data): onset in beats
    length: float     # in beats: where the next clip starts

    def __rshift__(self, other):          # then 
        moved = [(onset + self.length, data) for onset, data in other.events]
        return Clip(self.events + moved, self.length + other.length)

    def __mul__(self, times):             # again
        out = Clip([], 0)
        for _ in range(times):
            out = out >> self
        return out

    def __add__(self, n):
        return Clip([(onset, data + n) for onset, data in self.events], self.length)

    # note(0) * 3 => 1st it: out = Clip([], 0) >> note(0) → Clip([(0, 0)], 1) ...

    def __or__(self, other):              # together
        return Clip(self.events + other.events, max(self.length, other.length))

def note(data, length=1):
    return Clip([(0, data)], length)

def rest(length):
    return Clip([], length)

class Scale:
    def __init__(self, steps, root):
        self.root = root
        self.distances = []
        distance = 0
        for step in steps:
            self.distances.append(distance)
            distance += step

    def __getitem__(self, degree):
        octave = degree // len(self.distances)
        step = degree % len(self.distances)
        return self.root + (12 * octave) + self.distances[step]

def play_clip(clip, scale, beat_s=0.5):
    sounds = [(onset * beat_s, pluck(scale[data].f)) for onset, data in clip.events]
    return play(mix_sounds(sounds))

a_major = Scale([2, 2, 1, 2, 2, 2, 1], Pitch("A4"))

In [4]:
class Sine:
    def __init__(self, f, amp=0.2):
        self.f = f
        self.amp = amp
        self.phase = 0.0

    def block(self, frames):
        phases = self.phase + 2 * np.pi * self.f * np.arange(frames) / SR
        self.phase = (self.phase + 2 * np.pi * self.f * frames / SR) % (2 * np.pi)
        return self.amp * np.sin(phases)

class Square:
    def __init__(self, f, amp=0.1):
        self.f = f
        self.amp = amp
        self.phase = 0.0

    def block(self, frames):
        phases = self.phase + 2 * np.pi * self.f * np.arange(frames) / SR
        self.phase = (self.phase + 2 * np.pi * self.f * frames / SR) % (2 * np.pi)
        return self.amp * np.sign(np.sin(phases))

class Voice:
    def __init__(self, wave):
        self.wave = wave
        self.i = 0

    def block(self, frames):
        out = np.zeros(frames)
        chunk = self.wave[self.i : self.i + frames]
        out[:len(chunk)] = chunk
        self.i += frames
        return out

    @property
    def done(self):
        return self.i >= len(self.wave)

class Mixer:
    def __init__(self, sources=None):
        self.sources = sources or []

    def block(self, frames):
        out = np.zeros(frames)
        for source in list(self.sources):
            out += source.block(frames)
            if getattr(source, "done", False): # False if the attribute doesn'e exist
                self.sources.remove(source)
        return out

def stream_of(source):
    def callback(outdata, frames, time, status):
        outdata[:, 0] = np.clip(source.block(frames), -1, 1)
    return sd.OutputStream(samplerate=SR, channels=1, blocksize=512, callback=callback)

out = None

def live(source):                            # play a source, stopping what played before
    global out
    if out is not None:
        out.stop()
    out = stream_of(source)
    out.start()

def stop():
    if out is not None:
        out.stop()

## A · Warm-up

**A1.** Play an A major triad with three sines in a mixer: degrees 0, 2 and 4 of `a_major`. Then, while it plays, move the top note up to degree 5.

In [ ]:
# a_major = Scale([2, 2, 1, 2, 2, 2, 1], Pitch("A4"))

chord = Mixer([Sine(a_major[d].f, 0.1) for d in [0, 2, 4]])
live(chord)

In [86]:
chord.sources[2].f = a_major[5].f

**A2.** Write `Octaves(f)`: a source that plays `f` and `2 * f` together. It holds two sines, and its `block` sums theirs.

It is a source made of sources, like the mixer, but its parts are fixed.

In [77]:
class Octaves:
    def __init__(self, f):
        self.low = Sine(f, 0.15)
        self.high = Sine(2 * f, 0.08)

    def block(self, frames):
        return self.low.block(frames) + self.high.block(frames)

live(Octaves(110))

In [78]:
live(Octaves(400))

In [87]:
stop()

## B · Waveforms

A sine, a square, a saw and a triangle differ only in what they do with the phase. Let `p` be the position in the cycle, from 0 to 1:

```python
p = (phases / (2 * np.pi)) % 1
```

| wave | from `p` |
|---|---|
| saw | `2 * p - 1`: up from -1 to 1, then a jump |
| triangle | `2 * abs(saw) - 1`, with `saw = 2 * p - 1` |
| pulse | `1` while `p < width`, else `-1` |

**B1.** Write `Saw(f, amp=0.1)`, in the style of `Square`: the same phase, a different last line. Listen at 110 Hz, then at 3000 Hz.

In [8]:
class Saw:
    def __init__(self, f, amp=0.1):
        self.f = f
        self.amp = amp
        self.phase = 0.0

    def block(self, frames):
        phases = self.phase + 2 * np.pi * self.f * np.arange(frames) / SR
        self.phase = (self.phase + 2 * np.pi * self.f * frames / SR) % (2 * np.pi)
        p = (phases / (2 * np.pi)) % 1
        return self.amp * (2 * p - 1)

low = Saw(110)
live(low)

In [9]:
stop()

At high notes the saw sounds out of tune, with whistles that do not belong to the note. Its jump has harmonics far above `SR / 2`, and they fold back as other frequencies: **aliasing**. On low notes it is the sound of old sound chips.

**B2.** Four sources that differ in one line are one source with a parameter. Write `Osc(f, shape, amp=0.1)`, where `shape` is a function from the phases to the samples, and the four shapes as functions.

Then change `osc.shape` while it plays.

In [44]:
def sine(phases):
    return np.sin(phases)

def square(phases):
    return np.sign(np.sin(phases))

def saw(phases):
    return 2 * ((phases / (2 * np.pi)) % 1) - 1

def triangle(phases):
    return 2 * np.abs(saw(phases)) - 1

class Osc:
    def __init__(self, f, shape=sine, amp=0.1):
        self.f = f
        self.shape = shape
        self.amp = amp
        self.phase = 0.0

    def block(self, frames):
        phases = self.phase + 2 * np.pi * self.f * np.arange(frames) / SR
        self.phase = (self.phase + 2 * np.pi * self.f * frames / SR) % (2 * np.pi)
        return self.amp * self.shape(phases)

osc = Osc(220)
live(osc)

In [41]:
stop()

In [40]:
for shape in [sine, triangle, square, saw] * 2:
    osc.shape = shape
    await asyncio.sleep(1)

The shapes are pure functions: same phases in, same samples out. The state, the phase, stays in `Osc`.

**B3.** Write `pulse(width)`: it gives back a shape, a function of the phases, that is 1 while `p < width` and -1 after. `pulse(0.5)` is a square; try `pulse(0.1)`, thin and nasal.

In [46]:
def pulse(width):
    def shape(phases):
        p = (phases / (2 * np.pi)) % 1
        return np.where(p < width, 1.0, -1.0)
    return shape

osc.shape = pulse(0.1)

In [51]:
stop()

## C · Noise with a memory

White noise needs no memory: every sample is new.

**C1.** Write `Noise(amp=0.1)`, white noise between `-amp` and `amp`. Use `rng.uniform`.

In [48]:
class Noise:
    def __init__(self, amp=0.1):
        self.amp = amp

    def block(self, frames):
        return self.amp * rng.uniform(-1, 1, frames)

live(Noise())

**C2.** A softer noise: each sample is the average of the last `k` white samples. The high frequencies cancel out, and it sounds darker: like wind or the sea.

`np.convolve(x, np.ones(k) / k, mode="valid")` gives the average of every `k` neighbours in `x`, so `k - 1` fewer samples than `x`. The first samples of a block need the last `k - 1` of the block before: keep them in `self.tail`.

In [52]:
class Soft:
    def __init__(self, k=16, amp=0.3):
        self.k = k
        self.amp = amp
        self.tail = np.zeros(k - 1)          # the last k - 1 white samples

    def block(self, frames):
        x = np.concatenate([self.tail, rng.uniform(-1, 1, frames)])
        self.tail = x[-(self.k - 1):]
        return self.amp * np.convolve(x, np.ones(self.k) / self.k, mode="valid")

soft = Soft()
live(soft)

**C3.** Change `soft.k` while it plays: 2, 16, 64. What happens with `k = 64`, and why does it get quieter? What would you hear if every block started from `np.zeros(k - 1)` instead of `self.tail`?

In [53]:
# k = 64: only slow movements survive the average, so it is darker, and averaging many
# independent samples cancels most of them out: quieter.
# Starting every block from zeros: the first k - 1 samples of each block fade in from
# silence, 94 times a second: a buzz, as with the sine that forgot its phase.
soft.k = 64
soft.tail = np.zeros(soft.k - 1)

In [84]:
stop()

## D · Effects

An effect is a source that holds another source: it asks it for a block, changes it, and gives it back.

**D1.** Write `Gain(source, g)`: the source, times `g`. Then make a sine louder and softer while it plays.

In [ ]:
class Gain:
    def __init__(self, source, g):
        self.source = source
        self.g = g

    def block(self, frames):
        return self.g * self.source.block(frames)

louder = Gain(Sine(220), 0.5)
live(louder)

In [81]:
louder.g = 1.5

**D2.** Write `Drive(source, amount)`: `np.tanh(amount * x)`. `tanh` is close to `x` near zero and flattens towards ±1: a soft clip, like an overdriven amplifier. Try it on a sine with `amount` 1, 5 and 20, and listen to the sine turn into a square.

In [66]:
class Drive:
    def __init__(self, source, amount=5):
        self.source = source
        self.amount = amount

    def block(self, frames):
        return 0.2 * np.tanh(self.amount * self.source.block(frames))

driven = Drive(Sine(110, 1.0))
live(driven)

In [65]:
driven.amount = 20

**D3.** An echo plays what came out `delay_s` seconds ago, softer, on top of what comes in now. It needs a memory of the last `delay_s` seconds of output: a **delay line**.

Keep the output in `self.buffer`, an array of `round(delay_s * SR)` samples, oldest first. For each block:

1. `out = dry + feedback * self.buffer[:frames]`: the oldest samples are the echo;
2. drop the oldest `frames` samples from the buffer, and append `out`.

Because `out` goes back into the buffer, the echo has an echo: `feedback` decides how fast they fade. This works as long as the delay is longer than a block.

In [68]:
class Echo:
    def __init__(self, source, delay_s=0.3, feedback=0.5):
        self.source = source
        self.feedback = feedback
        self.buffer = np.zeros(round(delay_s * SR))     # the last delay_s seconds of output

    def block(self, frames):
        dry = self.source.block(frames)
        out = dry + self.feedback * self.buffer[:frames]
        self.buffer = np.concatenate([self.buffer[frames:], out])
        return out

m = Mixer()
live(Echo(m))

In [71]:
m.sources.append(Voice(pluck(Pitch("E5").f, dur_s=0.3)))       # run it again and again

In [72]:
for d in [0, 2, 4, 7, 4, 2]:
    m.sources.append(Voice(pluck(a_major[d].f, dur_s=0.3)))
    await asyncio.sleep(0.15)

Try `feedback=0.9`, and `delay_s=0.05`: very short echoes stop being echoes and colour the sound, like a metal tube.

In [73]:
stop()

## E · Parameters that move

In the sources above, `f` and `amp` are numbers. If they can also be sources, one source can move another: a slow sine moving the frequency of a fast one is a vibrato.

**E1.** Write `values(x, frames)`: if `x` has a `block` method, give back `x.block(frames)`; otherwise an array of `frames` copies of `x`. Use `hasattr` and `np.full`.

In [31]:
def values(x, frames):
    if hasattr(x, "block"):
        return x.block(frames)
    return np.full(frames, float(x))

values(3, 4), values(Sine(1000), 4)

(array([3., 3., 3., 3.]),
 array([0.        , 0.02610524, 0.05176381, 0.07653669]))

**E2.** With a frequency that changes at every sample, the phase no longer grows by the same step: it grows by `2π · f / SR` at each sample, with that sample's `f`. A running sum, `np.cumsum`, adds the steps up.

Write `Osc2(f, shape=sine, amp=0.1)`, like `Osc` but with `f` and `amp` read through `values`. Then write `LFO(rate, depth, centre)`, a low-frequency oscillator: `centre + depth * sin`. It is a source too.

In [32]:
class Osc2:
    def __init__(self, f, shape=sine, amp=0.1):
        self.f = f
        self.shape = shape
        self.amp = amp
        self.phase = 0.0

    def block(self, frames):
        steps = 2 * np.pi * values(self.f, frames) / SR
        phases = self.phase + np.cumsum(steps)
        self.phase = phases[-1] % (2 * np.pi)
        return values(self.amp, frames) * self.shape(phases)

class LFO:
    def __init__(self, rate, depth, centre):
        self.wave = Sine(rate, depth)
        self.centre = centre

    def block(self, frames):
        return self.centre + self.wave.block(frames)

vibrato = LFO(rate=5, depth=6, centre=330)
lead = Osc2(f=vibrato, shape=triangle)
live(lead)

**E3.** Play with it while it sounds: a siren (slow and wide), a tremolo (the amplitude moved by another LFO), and a pulse whose width is moved by an LFO: **PWM**, the sound of old analog pads. For the last one, the width must be read through `values` too.

In [33]:
vibrato.wave.f = 0.3
vibrato.wave.amp = 80                                    # a siren

In [34]:
lead.amp = LFO(rate=6, depth=0.08, centre=0.1)          # a tremolo

In [35]:
def pwm(width):
    def shape(phases):
        p = (phases / (2 * np.pi)) % 1
        return np.where(p < values(width, len(phases)), 1.0, -1.0)
    return shape

pad_voice = Osc2(f=110, shape=pwm(LFO(rate=0.5, depth=0.4, centre=0.5)), amp=0.08)
live(pad_voice)

In [36]:
stop()

## F · No clicks

A click is a jump in the wave: something in the state changed all at once. Each fix spreads the change over a few milliseconds.

In [37]:
s = Sine(330)
live(s)

In [38]:
s.amp = 0.02        # a jump in amplitude: listen for the click

**F1.** Write `Smooth(value)`: a source with a `target`. Each block it walks from its current value to the target, with `np.linspace`, then stays there. Use it as the amplitude of an `Osc2`, and change `target`: no click.

In [39]:
class Smooth:
    def __init__(self, value):
        self.value = value
        self.target = value

    def block(self, frames):
        ramp = np.linspace(self.value, self.target, frames)
        self.value = self.target
        return ramp

volume = Smooth(0.2)
live(Osc2(330, amp=volume))

In [40]:
volume.target = 0.02

**F2.** Jumping to another point of a voice clicks too. Give `Voice` a method `jump(i)`, and a crossfade: for one block, play both the old position, fading out, and the new one, fading in.

Write it as `Voice2`, with a helper `read(i, frames)` that gives `frames` samples from `i`, padded with zeros.

In [41]:
class Voice2:
    def __init__(self, wave):
        self.wave = wave
        self.i = 0
        self.old = None                      # where we jumped from, for one block

    def read(self, i, frames):
        out = np.zeros(frames)
        chunk = self.wave[i : i + frames]
        out[:len(chunk)] = chunk
        return out

    def jump(self, i):
        self.old = self.i
        self.i = i

    def block(self, frames):
        out = self.read(self.i, frames)
        if self.old is not None:
            ramp = np.linspace(0, 1, frames)
            out = ramp * out + (1 - ramp) * self.read(self.old, frames)
            self.old = None
        self.i += frames
        return out

    @property
    def done(self):
        return self.i >= len(self.wave)

Try it on a remix: Fra Martino, jumping to a random beat every beat, then a roll that repeats the start faster and faster. Compare `v.jump(...)` with `v.i = ...`.

In [42]:
call = note(0) >> note(1) >> note(2) >> note(0)
fm = (call >> note(2) >> note(3) >> note(4, 2)) * 2
v = Voice2(mix_sounds([(onset * 0.5, pluck(a_major[d].f)) for onset, d in fm.events]))
live(v)

beat = round(0.5 * SR)
for _ in range(8):
    v.jump(beat * rng.integers(0, 8))
    await asyncio.sleep(0.5)
for n in [1, 2, 4, 8, 16]:
    for _ in range(n):
        v.jump(0)
        await asyncio.sleep(0.5 / n)
v.jump(0)

In [43]:
stop()

## G · A minute without stopping

Perform for a minute, with these rules:

- **one stream**, started at the beginning, stopped at the end;
- **no `stop()` in between**: everything changes while it plays, one cell at a time;
- **at least three gestures** among: a change of shape or frequency, an LFO, an effect added or removed, notes played into a mixer, a remix with `jump`.

A start: a mixer inside an echo, with a drone in it. Then add, change, remove.

In [44]:
stage = Mixer()
live(Echo(stage, delay_s=0.4, feedback=0.4))
stage.sources.append(Osc2(55, shape=triangle, amp=0.1))

In [45]:
# your performance here

While you play, notice what `await asyncio.sleep` cannot do: a rhythm made with it wobbles, and two gestures cannot run at the same time from one cell. Both point to the same missing piece: a clock that the sources themselves can read.

## H · Extensions

**H1 · A load meter.** The callback has `512 / SR` seconds, about 10.7 ms, to answer. Write `Metered(source)`: a source that times `source.block` with `time.perf_counter` and keeps the last and the largest share of the budget in `self.last` and `self.peak`. Then rain voices into a mixer until the cracks start.

In [46]:
import time as clock

class Metered:
    def __init__(self, source):
        self.source = source
        self.last = 0.0
        self.peak = 0.0

    def block(self, frames):
        t0 = clock.perf_counter()
        out = self.source.block(frames)
        self.last = (clock.perf_counter() - t0) / (frames / SR)
        self.peak = max(self.peak, self.last)
        return out

rain = Mixer()
meter = Metered(rain)
live(meter)
w = pluck(440, dur_s=2)
for _ in range(200):
    rain.sources.append(Voice(w))
await asyncio.sleep(0.5)
print(f"{len(rain.sources)} voices · load {meter.last:.0%} · peak {meter.peak:.0%}")

200 voices · load 4% · peak 10%


In [47]:
stop()

**H2 · An echo that ends.** In a mixer, `Echo(Voice(...))` never goes away: it has no `done`. Give it one: done when its source is done and the buffer is all but silent (`np.abs(self.buffer).max() < 1e-4`).

**H3 · Stereo.** Open the stream with `channels=2` and write a `Pan(source, position)` that gives back two columns. What has to change in `stream_of` and in `Mixer`?